In [1]:
import numpy as np

from sklearn.datasets import load_diabetes

from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score

from sklearn.linear_model import LinearRegression

In [2]:
x, y = load_diabetes(return_X_y = True, as_frame = True)

print(x.head())
print()
print()

print(y.head())

        age       sex       bmi        bp        s1        s2        s3  \
0  0.038076  0.050680  0.061696  0.021872 -0.044223 -0.034821 -0.043401   
1 -0.001882 -0.044642 -0.051474 -0.026328 -0.008449 -0.019163  0.074412   
2  0.085299  0.050680  0.044451 -0.005670 -0.045599 -0.034194 -0.032356   
3 -0.089063 -0.044642 -0.011595 -0.036656  0.012191  0.024991 -0.036038   
4  0.005383 -0.044642 -0.036385  0.021872  0.003935  0.015596  0.008142   

         s4        s5        s6  
0 -0.002592  0.019907 -0.017646  
1 -0.039493 -0.068332 -0.092204  
2 -0.002592  0.002861 -0.025930  
3  0.034309  0.022688 -0.009362  
4 -0.002592 -0.031988 -0.046641  


0    151.0
1     75.0
2    141.0
3    206.0
4    135.0
Name: target, dtype: float64


In [3]:
# Print shapes

print(x.shape)
print(y.shape)

(442, 10)
(442,)


In [4]:
# TrainTest Split
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size = 0.2, random_state = 1)

# Creating our `BatchGD` class

In [5]:
class BatchGD:
    def __init__(self, lr = 0.01, epochs = 100):
        self.coef_ = None
        self.intercept_ = None
        self.lr = lr
        self.epochs = epochs
        self.coef_derivatives = None

    def fit(self, x_train, y_train):
        
        # Initialize intercept(B0) = 0 and 
        # Bn = 1
        self.intercept_ = 0
        self.coef_ = np.ones(x_train.shape[1])    # We have 10 ccolumns, so we will have 10 slopes/co-effecients

        for i in range(self.epochs):
            # Calculating y_hat
            y_hat = np.dot(x_train, self.coef_) + self.intercept_
            intercept_derivative = (-2) * np.mean(y_train - y_hat)
            coef_derivatives = -2 * np.dot((y_train - y_hat),x_train)/x_train.shape[0]

            # Updating coefs, intercept and derivative(For monitoring)
            self.coef_ = (self.coef_) - (self.lr)*(coef_derivatives)
            self.intercept_ = (self.intercept_) - (self.lr)*(intercept_derivative)
            self.coef_derivatives = coef_derivatives
            

            print(f"""{'='*15} Epoch{i} {'='*15}
    Derivatives/Slopes: {coef_derivatives}
    
    Coeffecients: {self.coef_}
    
    Intercept: {self.intercept_}
    
    """)
            
        print(f"""{'='*15} Final Result {'='*15}
        Derivatives/Slope: {self.coef_derivatives}
        Coeffecients: {self.coef_}
        Intercept: {self.intercept_}
        """)      

    def predict(self, x_test):
        return np.dot(x_test, self.coef_) + self.intercept_

In [6]:
x_train.shape[1]

10

# ChatGPT Version: It said this code is correct one, but I got similar result in both codes

In [17]:
class BatchGD_2:
    def __init__(self, lr = 0.01, epochs = 100):
        self.coef_ = None
        self.intercept_ = None
        self.lr = lr
        self.epochs = epochs
        self.coef_derivatives = None

    def fit(self, x_train, y_train):
        
        n_samples, n_features = x_train.shape

        # Initialize parameters
        self.intercept_ = 0.0
        self.coef_ = np.ones(n_features)

        for i in range(self.epochs):
            # Prediction
            y_hat = np.dot(x_train, self.coef_) + self.intercept_

            # Error
            error = y_train - y_hat

            # Gradients (correct form)
            intercept_derivative = -2 * np.mean(error)
            coef_derivative = -2 * np.dot(x_train.T, error) / n_samples

            # Update, coef, intercept and derivatives(For monitoring)
            self.coef_ -= self.lr * coef_derivative
            self.intercept_ -= self.lr * intercept_derivative
            self.coef_derivatives = coef_derivatives


            # Optional: track loss
            loss = np.mean(error ** 2)

            print(f"""{'='*15} Epoch{i} {'='*15}
    Derivatives/Slopes: {coef_derivative}
    
    Co-effecients: {self.coef_}
    
    Intercept: {self.intercept_}
    Loss: {loss}
    {'-'*20}
    
    """)
            
        print(f"""{'='*15} Final Result {'='*15}
    Derivatives/Slope: {self.coef_derivatives}
    Coeffecients: {self.coef_}
    Intercept: {self.intercept_}
    """)  

    def predict(self, x_test):
        return np.dot(x_test, self.coef_) + self.intercept_

In [7]:
bgd = BatchGD(
    lr = 0.1,
    epochs = 6000
)

bgd.fit(x_train, y_train)

=============== Epoch0 ===============
    Derivatives/Slopes: [-1.44494852e+00 -1.96989393e-03 -5.03396025e+00 -3.58515777e+00
 -1.73011893e+00 -1.37448938e+00  3.23650002e+00 -3.56684319e+00
 -4.52640415e+00 -3.42986257e+00]

    Coeffecients: [1.14449485 1.00019699 1.50339603 1.35851578 1.17301189 1.13744894
 0.67635    1.35668432 1.45264041 1.34298626]

    Intercept: 30.674126042563042

    
=============== Epoch1 ===============
    Derivatives/Slopes: [-1.44940964 -0.03789246 -4.92048083 -3.5276704  -1.6755413  -1.3189186
  3.17841983 -3.48638923 -4.47566621 -3.32514049]

    Coeffecients: [1.28943582 1.00398624 1.99544411 1.71128282 1.34056602 1.2693408
 0.35850802 1.70532324 1.90020704 1.67550031]

    Intercept: 55.21282653753392

    
=============== Epoch2 ===============
    Derivatives/Slopes: [-1.45230391 -0.06616365 -4.82857582 -3.48072133 -1.63104385 -1.27366927
  3.13103357 -3.42077943 -4.43379353 -3.24029395]

    Coeffecients: [1.43466621 1.0106026  2.47830169 2.059

# Let's compare LinearRegression and BatchGD results

In [20]:
lr = LinearRegression()

lr.fit(x_train, y_train)

y_pred_lr = lr.predict(x_test)
y_pred_bgd = bgd.predict(x_test)

r2_lr = r2_score(y_test, y_pred_lr)
r2_bgd = r2_score(y_test, y_pred_bgd)

print(f"""{'='*15} LinearRegression {'='*15}
-> Co-effecients: {lr.coef_}
-> Intercept: {lr.intercept_}
-> R2_Score: {r2_lr}

{'='*15} BatchGD {'='*15}
-> Co-effecients: {bgd.coef_}
-> Intercept: {bgd.intercept_}
-> R2_Score: {r2_bgd}""")



=============== LinearRegression ===============
-> Co-effecients: [ -30.62219046 -272.2506057   528.85252681  327.69716891 -581.01973841
  332.97061365  -27.97314298  139.28036115  665.07667773   61.90517166]
-> Intercept: 151.4300932966009
-> R2_Score: 0.4384316213369278

=============== BatchGD ===============
-> Co-effecients: [ -13.55759965 -240.58663068  515.28761177  307.45684356  -43.90552869
 -105.37195244 -228.74868485  130.71005193  412.96138983  101.23062227]
-> Intercept: 151.37456795194842
-> R2_Score: 0.4323357987947498


In [9]:
np.mean((y_pred_lr - y_pred_bgd)**2)

19.816657758683753

In [14]:
# See difference in the prediction

print(y_pred_lr-y_pred_bgd)
# The difference in the prediction by both the model is very less
print()
print()

# Calculating overall difference
print(np.mean((y_pred_lr - y_pred_bgd)**2))    # Even this is very low


[  3.49964714   6.03198318  -1.69260007   0.27355911  -0.79031565
   2.95126111   0.89346519  -5.6772722    4.25157214   1.43144852
   7.45044773   4.92808291  -2.35176464  -1.93121168   8.30788461
   1.94839853   2.16711941   0.47323847  -9.28669006   1.44600888
   0.79070763  -4.36367489   1.44026328   5.44630974   2.61924892
   5.09996907  -2.2984648    4.90574734  -4.47052898   1.09651536
  -2.42139822  -1.00215398   2.93050206   2.00501785   9.65552868
   0.71258292   6.65565496   4.86402894  -0.20951362   3.5572178
   3.48077928   2.24016047  -0.55220144  -2.34938414   5.05767514
   3.18442702   3.86623133   5.08665509   2.13378079  -8.81734501
 -10.14854519  -4.39830444  -3.92309453   4.39590674   1.53887477
  -2.00588869   1.7180564   -9.56959572  -7.57828936  -6.94390324
   0.11535051   4.06972011   3.26017879   0.28812105   0.39493186
  -2.78136755  -5.60780555  -4.70349692   4.37267035  -4.12893557
 -13.2811405   -4.34247088  -1.30747493   0.64038877   0.52182945
   6.200653

In [16]:
x_train.corr()

,age,sex,bmi,bp,s1,s2,s3,s4,s5,s6
age,1.000000,0.160236,0.242503,0.345964,0.286700,0.232955,-0.079775,0.222736,0.298518,0.288280
sex,0.160236,1.000000,0.077537,0.234364,0.021532,0.119450,-0.380296,0.331263,0.157530,0.210015
bmi,0.242503,0.077537,1.000000,0.427926,0.222435,0.210712,-0.354172,0.390201,0.461354,0.389494
bp,0.345964,0.234364,0.427926,1.000000,0.240943,0.176072,-0.181745,0.262383,0.398141,0.391905
s1,0.286700,0.021532,0.222435,0.240943,1.000000,0.898966,0.073968,0.533274,0.503481,0.310577
s2,0.232955,0.119450,0.210712,0.176072,0.898966,1.000000,-0.151303,0.636579,0.303717,0.265432
s3,-0.079775,-0.380296,-0.354172,-0.181745,0.073968,-0.151303,1.000000,-0.732883,-0.421035,-0.288735
s4,0.222736,0.331263,0.390201,0.262383,0.533274,0.636579,-0.732883,1.000000,0.630236,0.425137
s5,0.298518,0.157530,0.461354,0.398141,0.503481,0.303717,-0.421035,0.630236,1.000000,0.484219
s6,0.288280,0.210015,0.389494,0.391905,0.310577,0.265432,-0.288735,0.425137,0.484219,1.000000


# Why are coefficients very different even though R² score is almost the same?

You observed:

- Coefficients from Gradient Descent (BatchGD) and Linear Regression are **very different**
- But the **R² score is almost the same**

This feels confusing at first—but it's actually a very important concept in machine learning.

---

# 🧠 Simple Idea

> Different coefficients can still produce almost the same predictions.

And since **R² depends on predictions (not coefficients)**, the score remains similar.

---

# 📊 Your Correlation Matrix (Important Part)

Here are some key correlations from your data:

| Feature Pair | Correlation |
|-------------|------------|
| s1 ↔ s2     | **0.898** 🔥 |
| s3 ↔ s4     | **-0.732** 🔥 |
| s4 ↔ s5     | **0.630** |
| s1 ↔ s4     | **0.533** |

---

# 🚨 What does this mean?

- Values close to **+1 or -1** = strong relationship
- Your dataset has **strongly correlated features**

👉 Example:
- `s1` and `s2` are almost telling the same story
- `s3` and `s4` are strongly (negatively) related

---

# 🤔 Why does this affect coefficients?

Imagine this situation:
`s1 ≈ s2`


The model now has **multiple ways** to make predictions:

### Option 1:
`Use s1 more, s2 less`


### Option 2:
`Use s2 more, s1 less`


👉 Both give **almost same prediction**

---

# 💥 Real Example from your output

### Linear Regression:
`s1: -581` <br>
`s2: 332`


### Gradient Descent:
`s1: -43` <br>
`s2: -105`


👉 Coefficients are very different  
👉 But predictions are similar → R² stays similar

---

# 🧠 Deep Intuition (in simple words)

Your model is doing this: <br>
Prediction = (s1 * w1) + (s2 * w2) + ...


Since `s1` and `s2` are similar:

- Increasing `w1` and decreasing `w2`
- OR decreasing `w1` and increasing `w2`

👉 can cancel out and give same result

---

# ⚖️ Why LinearRegression and GD differ

### Linear Regression (closed-form):
- Finds **exact mathematical solution**
- Picks one optimal set of coefficients

### Gradient Descent:
- Moves step-by-step
- Stops at a **slightly different point**
- Still gives similar predictions

---

# 🎯 Key Takeaways

- ✔ R² depends on **predictions**, not coefficients  
- ✔ Correlated features → **multiple valid coefficient combinations**  
- ✔ That’s why coefficients differ but performance stays similar  

---

# 💡 Simple Analogy

Think of it like:

> You and your friend take different routes to reach the same destination.

- Routes (coefficients) are different 🚗  
- Destination (prediction) is same 🎯  

---

# ✅ Final Conclusion

Your observation is **correct and expected**.

There is no bug in your code.

👉 The difference happens because:
- Your features are **correlated**
- The model can distribute weights in multiple ways
- Predictions remain almost the same → R² remains same

---

# 🎉 Session Complete